# Phase 7 · Submission S2 — Architecture C (Stacked LSTM) + Partial / Layer-wise Fine-tuning, contrastive bi-encoder, LP-FT

**Task:** Prior Case Retrieval. For every query case, rank the candidates of the same split's pool and return exactly 7 unique candidate ids, most relevant first. **Primary metric:** Micro-F1@7 (secondary Micro-P@7, Micro-R@7).

**Method.** The retrieval pipeline is the same as S1. Only the Phase-1 model and the adaptation technique differ.
1. **Backbone:** Phase-1 Architecture C (3-layer stacked LSTM, hidden 1280, 103.4M parameters).
2. **Adaptation, partial / layer-wise fine-tuning:** the embeddings and the two lower LSTM layers stay frozen. The **top LSTM layer and the final LayerNorm are fully trained** (≈ 13M parameters). Dropout is off everywhere.
3. **Input and long documents:**
   - The masked placeholders become single words (`<CITATION>` → "Citation", `<NAME>` → "Name", `<ORG>` → "Org").
   - Chunks are 510 BPE tokens, within C's 1,024-token Phase-1 training length.
   - The **first 4 + last 4 chunks** of each case are used.
4. **Bi-encoder:**
   - Each chunk becomes [mean ; final state] of its token states, standardised with frozen Phase-1 statistics.
   - A projection to 256 dimensions and L2 normalisation follow. The case vector is the normalised mean of its chunk vectors, and relevance is cosine similarity.
5. **The query is always removed from its own ranking** (every query is in its own pool). The top 7 of the rest are submitted.
6. **Loss:** InfoNCE (temperature 0.05) with 2 TF-IDF hard negatives per pair plus in-batch negatives. Other relevant cases of the same query are masked.
7. **LP-FT:**
   - **Step 1:** the projection is trained on cached frozen features ("epoch 0").
   - **Step 2:** the top LSTM layer, the final LayerNorm and the projection are fine-tuned, with 2 + 2 chunks per case in training and 4 + 4 for evaluation.
   - The best epoch on dev Micro-F1@7 is kept.
8. **No leakage:** train labels for training, dev labels for selection and analysis. TF-IDF is used only for hard negatives and as a reference, never in the submitted ranking.
9. **Analysis:**
   - frozen Phase-1 retrieval without training;
   - chunk selection × mean/max pooling;
   - dev score by number of relevant cases;
   - queries with no hit;
   - overlap with TF-IDF hits.
   The final case vectors go to `outputs/teacher_vectors.pt`, for comparisons and ensembling.

**Before running (Kaggle)**
- Accelerator: **GPU T4 ×2**. Add-ons → Secrets: **`HF`**. Internet on.
- Add Input → datasets: **`rohitbatra2004/phase133`** (Architecture C) and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- **Expected time ≈ 2.5–3.5 hours.** Checkpoints are saved every 300 steps and every epoch, and tokens and features are cached. To resume, add this notebook's previous output as an input and run again.

Outputs: `/kaggle/working/P7_S2_C_LSTM_LayerwiseFT/`. Upload `submission.zip`, and save the output folder as the Kaggle dataset **`Phase7_S2_Dataset`**.

In [1]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile, re, hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from sklearn.feature_extraction.text import TfidfVectorizer
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

PyTorch 2.11.0+cu128 | device: cuda | GPUs visible: 2


In [2]:
# Cell 2 — Paths and hyperparameters for this submission.
# Every setting lives here; no other cell hard-codes a value.
RUN_NAME = "P7_S2_C_LSTM_LayerwiseFT"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"
K = 7                                       # answers per query (fixed by the task)

# Long documents: chunks of 510 BPE tokens (+ [BOS], [EOS] = 512). A setting = (which chunks, how many).
CHUNK_TOKENS = 510
KEEP_CHUNKS = 8                             # the first 8 and last 8 chunks of every case are tokenised and kept
CACHE_SETTING = ("firstlast", 16)           # frozen features are computed for these chunks (analysis of other settings)
EVAL_SETTING = ("firstlast", 8)             # first 4 + last 4 chunks: case vectors for selection, analysis and test
TRAIN_SETTING = ("firstlast", 4)            # first 2 + last 2 chunks per case while fine-tuning (memory)
SECTION_SETTINGS = (("first", 1), ("first", 4), ("last", 4), ("firstlast", 8), ("firstlast", 16))

# Partial / layer-wise fine-tuning: train only the top LSTM layer (+ final LayerNorm); everything below stays frozen.
TUNED_LSTM_LAYERS = 1

# Bi-encoder head and contrastive loss.
PROJ_DIM = 256
FEATURE_CLAMP = 10.0                        # standardised chunk features clipped to ±10
TEMPERATURE = 0.05
HARD_POOL = 50                              # hard negatives are sampled from the 50 most TF-IDF-similar non-relevant cases
HARD_PER_PAIR = 2                           # hard negatives per (query, relevant case) pair

# Step 1 — projection trained on cached frozen features.
PROBE_EPOCHS = 30                           # evaluated every 3 epochs; the best on dev is kept
PROBE_LR = 1e-3
PROBE_PAIRS = 64                            # (query, relevant case) pairs per step

# Step 2 — top LSTM layer + projection fine-tuning.
EPOCHS = 3
PAIRS_PER_STEP = 4                          # 4 pairs × (query + relevant + 2 hard negatives) × 4 chunks = 64 chunks / step
LR_ADAPTER = 1e-4
LR_HEAD = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 300
CHECK_EVERY_STEPS = 600                     # quick dev evaluation during an epoch
EVAL_CHUNK_BATCH = 48                       # chunks per forward pass when encoding documents
USE_AMP = DEVICE.type == "cuda"
print("Run:", RUN_NAME, "| outputs:", WORK_DIR)

Run: P7_S2_C_LSTM_LayerwiseFT | outputs: /kaggle/working/P7_S2_C_LSTM_LayerwiseFT


In [3]:
# Cell 3 — Download Phase_7_PCR from Hugging Face using the Kaggle secret "HF".
from kaggle_secrets import UserSecretsClient
snapshot_download(repo_id=HF_REPO_ID, repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_7_PCR/**"])
P7_DIR = DATA_DIR / "Phase_7_PCR"
FILES = {"train": ("Train/train.json", "Train/train_candidates.json"), "dev": ("Dev/dev.json", "Dev/dev_candidates.json"),
         "test": ("Test/test.json", "Test/test_candidates.json")}
for q_file, c_file in FILES.values():
    assert (P7_DIR / q_file).is_file() and (P7_DIR / c_file).is_file(), f"Missing {q_file} or {c_file}"
print("Phase 7 data ready in", P7_DIR)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Phase 7 data ready in /kaggle/working/data/Phase_7_PCR


In [4]:
# Cell 4 — Load queries and candidate pools, replace the masked placeholders, and register one document per text.
# Queries are also in their own pool (data analysis); a query is encoded once and is never ranked for itself.
PLACEHOLDERS = ((re.compile(r"<\s*citation\s*>", re.I), " Citation "), (re.compile(r"<\s*name\s*>", re.I), " Name "),
                (re.compile(r"<\s*org\s*>", re.I), " Org "))

def normalise(sentences):
    text = " ".join(sentences) if isinstance(sentences, list) else str(sentences)
    for pattern, replacement in PLACEHOLDERS:
        text = pattern.sub(replacement, text)
    return " ".join(text.split())

QUERIES, POOL_IDS, DOC_TEXT, DOC_INDEX = {}, {}, {}, {}
for split, (q_file, c_file) in FILES.items():
    queries, pool = json.load(open(P7_DIR / q_file)), json.load(open(P7_DIR / c_file))
    docs = {}                                                   # document key -> text (pool first, then queries not in it)
    for cid, rec in pool.items():
        docs[str(cid)] = normalise(rec["text"] if isinstance(rec, dict) else rec)
    QUERIES[split] = {}
    for qid, rec in queries.items():
        qid, text = str(qid), normalise(rec["text"])
        key = qid if docs.get(qid) == text else f"Q:{qid}"      # same id and text as a pool document -> share it
        docs.setdefault(key, text)
        QUERIES[split][qid] = {"key": key, "rel": [str(c) for c in rec.get("relevant_candidates", [])]}
    POOL_IDS[split] = [str(c) for c in pool]
    DOC_TEXT[split] = list(docs.values())
    DOC_INDEX[split] = {k: i for i, k in enumerate(docs)}
    missing = sum(c not in DOC_INDEX[split] for v in QUERIES[split].values() for c in v["rel"])
    assert missing == 0, f"{split}: {missing} relevant ids are not in the pool"
    shared = sum(v["key"] == q for q, v in QUERIES[split].items())
    print(f"{split}: {len(QUERIES[split])} queries ({shared} also in the pool) | pool {len(POOL_IDS[split]):,} | "
          f"documents to encode {len(DOC_TEXT[split]):,} | labelled queries {sum(bool(v['rel']) for v in QUERIES[split].values())}")
TRAIN_QUERIES = [q for q, v in QUERIES["train"].items() if v["rel"]]
DEV_QUERIES = [q for q, v in QUERIES["dev"].items() if v["rel"]]
TEST_QUERIES = list(QUERIES["test"])
print("Example after normalisation:", DOC_TEXT["dev"][0][:300])

train: 827 queries (827 also in the pool) | pool 4,320 | documents to encode 4,320 | labelled queries 817
dev: 118 queries (118 also in the pool) | pool 1,023 | documents to encode 1,023 | labelled queries 118
test: 237 queries (237 also in the pool) | pool 1,727 | documents to encode 1,727 | labelled queries 0
Example after normalisation: Name , J. 1. This is an appeal from the decision of Org at Fort William in Bengal, dated November 25, 1924, by which the judgment of the Subordinate Judge of Asansol in Zillah Burdwan, dated February 5, 1923, was reversed and the suit dismissed with costs. The appellant prays that the decision of Or


In [5]:
# Cell 5 — Tokenise every document once (cased Phase-1 BPE) and keep its first and last KEEP_CHUNKS chunks (cached);
# chunk selection rules: "first" k, "last" k, "firstlast" k = first k/2 + last k/2 (all chunks if the case is shorter).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]"))
assert (PAD_ID, BOS_ID, EOS_ID) == (0, 2, 3)

def tokenise_split(texts):
    """For each document: {'n': total chunks, 'chunks': {chunk number: uint16 token array}} (first and last KEEP_CHUNKS)."""
    out = []
    for start in range(0, len(texts), 64):
        for enc in tokenizer.encode_batch(texts[start:start + 64]):
            ids = np.array(enc.ids, dtype=np.uint16)
            n = max(1, -(-len(ids) // CHUNK_TOKENS))
            keep = sorted(set(range(min(KEEP_CHUNKS, n))) | set(range(max(0, n - KEEP_CHUNKS), n)))
            out.append({"n": n, "chunks": {c: ids[c * CHUNK_TOKENS:(c + 1) * CHUNK_TOKENS] for c in keep}})
    return out

cache = WORK_DIR / "tokens_cache.pt"
if not cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/tokens_cache.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], cache)
if cache.exists():
    TOKENS = torch.load(cache, weights_only=False)
    print("Loaded cached tokens from", cache)
else:
    start_time = time.time()
    TOKENS = {split: tokenise_split(DOC_TEXT[split]) for split in DOC_TEXT}
    torch.save(TOKENS, cache)
    print(f"Tokenised {sum(len(v) for v in TOKENS.values()):,} documents in {(time.time() - start_time) / 60:.1f} min → {cache}")
for split in DOC_TEXT:
    assert len(TOKENS[split]) == len(DOC_TEXT[split])

def chosen(chunk_no, n, setting):
    """Is chunk `chunk_no` of a case with `n` chunks part of `setting`?"""
    kind, k = setting
    if kind == "first":
        return chunk_no < k
    if kind == "last":
        return chunk_no >= n - k
    return chunk_no < k // 2 or chunk_no >= n - k // 2           # "firstlast"

def doc_chunks(split, d, setting):
    """Token lists ([BOS] … [EOS]) of the chunks a setting uses for one document, in document order."""
    t = TOKENS[split][d]
    return [[BOS_ID] + t["chunks"][c].tolist() + [EOS_ID] for c in sorted(t["chunks"]) if chosen(c, t["n"], setting)]

for split in DOC_TEXT:
    n = np.array([t["n"] for t in TOKENS[split]])
    print(f"{split}: chunks per document median {int(np.median(n))}, p90 {int(np.percentile(n, 90))}, max {n.max()} | "
          f"fully covered by {EVAL_SETTING[1]} chunks: {100 * np.mean(n <= EVAL_SETTING[1]):.1f}%")

Tokenised 7,070 documents in 1.3 min → /kaggle/working/P7_S2_C_LSTM_LayerwiseFT/tokens_cache.pt
train: chunks per document median 9, p90 29, max 949 | fully covered by 8 chunks: 47.5%
dev: chunks per document median 10, p90 39, max 949 | fully covered by 8 chunks: 42.4%
test: chunks per document median 10, p90 32, max 949 | fully covered by 8 chunks: 44.1%


In [6]:
# Cell 6 — TF-IDF over each split's pool: hard negatives for every train query (top HARD_POOL non-relevant, never the query
# itself) and the TF-IDF reference score on dev. Neither is ever part of a submitted ranking.
def micro_at_k(ranked, gold):
    """Official metric: micro precision / recall / F1 over the first K answers; plus Recall@20/50/100."""
    hits = sum(len(set(r[:K]) & set(g)) for r, g in zip(ranked, gold))
    n_gold = sum(len(g) for g in gold)
    p, r = hits / (K * len(gold)), hits / max(n_gold, 1)
    out = {"micro_p@7": round(p, 4), "micro_r@7": round(r, 4), "micro_f1@7": round(2 * p * r / max(p + r, 1e-9), 4)}
    out |= {f"recall@{k}": round(sum(len(set(x[:k]) & set(g)) for x, g in zip(ranked, gold)) / max(n_gold, 1), 4) for k in (20, 50, 100)}
    return out

def rank_queries(scores, split, query_ids, depth=100):
    """scores [queries, documents] → for each query the top `depth` pool ids, without the query itself."""
    pool_rows = torch.tensor([DOC_INDEX[split][c] for c in POOL_IDS[split]])
    s = scores[:, pool_rows].clone().float()
    for i, q in enumerate(query_ids):
        own = QUERIES[split][q]["key"]
        if own in POOL_IDS[split]:
            s[i, POOL_IDS[split].index(own)] = -float("inf")
        if q in POOL_IDS[split]:
            s[i, POOL_IDS[split].index(q)] = -float("inf")
    top = s.topk(min(depth, s.size(1)), dim=1).indices.tolist()
    return [[POOL_IDS[split][j] for j in row] for row in top]

def gold(split, query_ids):
    return [QUERIES[split][q]["rel"] for q in query_ids]

def tfidf_scores(split, query_ids):
    vec = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z][a-z0-9]+\b", stop_words="english", min_df=2,
                          max_df=0.5, sublinear_tf=True, max_features=200_000, dtype=np.float32)
    D = vec.fit_transform(DOC_TEXT[split])
    Q = D[[DOC_INDEX[split][QUERIES[split][q]["key"]] for q in query_ids]]
    return torch.from_numpy((Q @ D.T).toarray())

hard_cache = WORK_DIR / "hard_negatives.pt"
if not hard_cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/hard_negatives.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], hard_cache)
if hard_cache.exists():
    HARD = torch.load(hard_cache, weights_only=False)
    print("Loaded cached hard negatives and TF-IDF reference from", hard_cache)
else:
    start_time = time.time()
    ranked = rank_queries(tfidf_scores("train", TRAIN_QUERIES), "train", TRAIN_QUERIES, depth=HARD_POOL + 60)
    negatives = {q: [c for c in r if c not in set(QUERIES["train"][q]["rel"])][:HARD_POOL] for q, r in zip(TRAIN_QUERIES, ranked)}
    tfidf_dev = micro_at_k(rank_queries(tfidf_scores("dev", DEV_QUERIES), "dev", DEV_QUERIES), gold("dev", DEV_QUERIES))
    tfidf_dev_ranked = rank_queries(tfidf_scores("dev", DEV_QUERIES), "dev", DEV_QUERIES, depth=K)
    HARD = {"negatives": negatives, "tfidf_dev": tfidf_dev, "tfidf_dev_top7": dict(zip(DEV_QUERIES, tfidf_dev_ranked))}
    torch.save(HARD, hard_cache)
    print(f"TF-IDF hard negatives for {len(negatives)} train queries in {time.time() - start_time:.0f} s")
print("TF-IDF reference on dev (not submitted):", HARD["tfidf_dev"])

TF-IDF hard negatives for 817 train queries in 21 s
TF-IDF reference on dev (not submitted): {'micro_p@7': 0.4007, 'micro_r@7': 0.3402, 'micro_f1@7': 0.368, 'recall@20': 0.4728, 'recall@50': 0.5827, 'recall@100': 0.668}


In [7]:
# Cell 7 — Rebuild the Phase-1 Architecture C backbone and strictly load its pretrained weights.
# Weights come from model_C_state_dict.pt; the exact configuration comes from model_C_metadata.json.
c_paths = sorted(glob.glob("/kaggle/input/**/architecture_C/model_C_state_dict.pt", recursive=True))
assert c_paths, "Add the dataset rohitbatra2004/phase133 as an input."
C_ROOT = Path(c_paths[0]).parent
C_METADATA = json.load(open(C_ROOT / "model_C_metadata.json"))
C_CONFIG = C_METADATA["model_config"]
assert C_CONFIG["hidden_size"] == C_CONFIG["embedding_dim"], "Phase-1 C used an identity hidden-to-embedding map"
HIDDEN = C_CONFIG["hidden_size"]

class ArchitectureCBackbone(nn.Module):
    """Phase-1 stacked LSTM language model without the vocabulary projection.
    Returns the normalised contextual state of every token: [batch, tokens, hidden_size].
    Padding is only at the end of a chunk, so the (left-to-right) states of real tokens are unaffected by it."""
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.lstm = nn.LSTM(cfg["embedding_dim"], cfg["hidden_size"], cfg["num_layers"],
                            dropout=cfg["dropout"], bias=cfg["use_bias"], batch_first=True)
        self.final_norm = nn.LayerNorm(cfg["hidden_size"])
        # Kept only so that the saved Phase-1 state dict loads strictly; unused here.
        self.output_bias = nn.Parameter(torch.zeros(cfg["vocab_size"]))

    def forward(self, input_ids):
        self.lstm.flatten_parameters()
        states, _ = self.lstm(self.token_embedding(input_ids))
        return self.final_norm(states)

saved = torch.load(c_paths[0], map_location="cpu", weights_only=True)
# The export is either the raw state dict or a dict that wraps it under "state_dict".
PHASE1_STATE = saved["state_dict"] if "state_dict" in saved else saved
del saved
check = ArchitectureCBackbone(C_CONFIG)
check.load_state_dict(PHASE1_STATE, strict=True)
n_backbone = sum(p.numel() for p in check.parameters())
assert n_backbone == int(C_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
del check
print(json.dumps(C_CONFIG, indent=2))
print(f"Phase-1 Architecture C weights verified from {C_ROOT} — {n_backbone:,} parameters")

{
  "vocab_size": 50000,
  "context_length": 1024,
  "embedding_dim": 1280,
  "hidden_size": 1280,
  "num_layers": 3,
  "dropout": 0.1,
  "tie_embeddings": true,
  "use_bias": true
}
Phase-1 Architecture C weights verified from /kaggle/input/datasets/rohitbatra2004/phase133/architecture_C — 103,404,880 parameters


In [8]:
# Cell 8 — Partial / layer-wise fine-tuning: unfreeze only the top LSTM layer and the final LayerNorm.
# The embeddings and the lower LSTM layers stay exactly as pretrained; LSTM dropout is set to 0 (dropout off everywhere).
def make_layerwise_backbone():
    bb = ArchitectureCBackbone(C_CONFIG)
    bb.load_state_dict(PHASE1_STATE, strict=True)
    bb.lstm.dropout = 0.0
    for p in bb.parameters():
        p.requires_grad_(False)
    tuned_layers = [C_CONFIG["num_layers"] - 1 - i for i in range(TUNED_LSTM_LAYERS)]
    for name, p in bb.named_parameters():
        if any(name.endswith(f"_l{layer}") for layer in tuned_layers) or name.startswith("final_norm"):
            p.requires_grad_(True)
    return bb, tuned_layers

# Check on the device: only the chosen layer and the final LayerNorm receive gradients.
probe_bb, tuned_layers = make_layerwise_backbone()
probe_bb = probe_bb.to(DEVICE).train()
probe_bb(torch.randint(4, 1000, (2, 16), device=DEVICE)).float().pow(2).mean().backward()
tuned = [n for n, p in probe_bb.named_parameters() if p.requires_grad]
assert all(probe_bb.get_parameter(n).grad is not None for n in tuned), "A tuned parameter got no gradient"
assert all(p.grad is None for p in probe_bb.parameters() if not p.requires_grad), "A frozen parameter got a gradient"
n_tuned = sum(p.numel() for p in probe_bb.parameters() if p.requires_grad)
print(f"Tuned LSTM layer(s) {tuned_layers} + final LayerNorm: {tuned}")
print(f"{n_tuned:,} trainable backbone parameters ({100 * n_tuned / n_backbone:.2f}% of the backbone)")
del probe_bb
torch.cuda.empty_cache()

Tuned LSTM layer(s) [2] + final LayerNorm: ['lstm.weight_ih_l2', 'lstm.weight_hh_l2', 'lstm.bias_ih_l2', 'lstm.bias_hh_l2', 'final_norm.weight', 'final_norm.bias']
13,120,000 trainable backbone parameters (12.69% of the backbone)


In [9]:
# Cell 9 — Bi-encoder. ChunkEncoder (split over both GPUs): chunk tokens → [mean ; final state] of the token states.
# Retriever: standardise (frozen-model statistics) → projection to PROJ_DIM → L2 norm → case vector = normalised mean.
class ChunkEncoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, ids):
        mask = ids.ne(PAD_ID)
        states = self.backbone(ids).float()
        m = mask.unsqueeze(-1).float()
        mean = (states * m).sum(1) / m.sum(1).clamp(min=1.0)
        last = states[torch.arange(states.size(0), device=states.device), mask.sum(1) - 1]
        return torch.cat([mean, last], dim=-1)                          # [chunks, 2·d_model]

class Retriever(nn.Module):
    def __init__(self, backbone, hidden, dim):
        super().__init__()
        self.encoder = ChunkEncoder(backbone)
        self.register_buffer("feat_mean", torch.zeros(2 * hidden))
        self.register_buffer("feat_std", torch.ones(2 * hidden))
        self.proj = nn.Linear(2 * hidden, dim)
        nn.init.orthogonal_(self.proj.weight); nn.init.zeros_(self.proj.bias)

    def chunk_vectors(self, raw):
        x = ((raw - self.feat_mean) / self.feat_std).clamp(-FEATURE_CLAMP, FEATURE_CLAMP)
        return F.normalize(self.proj(x), dim=-1)

def pool(vectors, lengths, mode="mean"):
    """Chunk vectors (grouped by document, in order) → one L2-normalised vector per document."""
    lengths = torch.as_tensor(lengths, device=vectors.device)
    out = torch.segment_reduce(vectors, mode, lengths=lengths, axis=0)
    return F.normalize(out, dim=-1)

torch.manual_seed(SEED)
model = Retriever(make_layerwise_backbone()[0], HIDDEN, PROJ_DIM).to(DEVICE)
N_GPUS = torch.cuda.device_count()
encoder = nn.DataParallel(model.encoder) if N_GPUS > 1 else model.encoder
TRAINABLE_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
SAVED_NAMES = TRAINABLE_NAMES + ["feat_mean", "feat_std"]
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_trainable:,} of {n_total:,} ({100 * n_trainable / n_total:.2f}%) | GPUs used: {max(N_GPUS, 1)}")

Trainable parameters: 13,775,616 of 104,060,496 (13.24%) | GPUs used: 2


In [10]:
# Cell 10 — Encoding helpers: padded chunk batches, raw chunk features of whole splits (frozen or adapted),
# and case vectors + dev evaluation (Micro-P/R/F1@7, query removed from its own ranking).
def chunk_batch(chunk_lists):
    T = max(len(x) for x in chunk_lists)
    ids = torch.full((len(chunk_lists), T), PAD_ID, dtype=torch.long)
    for i, x in enumerate(chunk_lists):
        ids[i, :len(x)] = torch.tensor(x)
    return ids.to(DEVICE)

@torch.no_grad()
def raw_features(split, setting, doc_ids=None):
    """Raw [mean ; final] features of every chosen chunk of the documents (in order) and the chunk count per document."""
    encoder.eval()
    doc_ids = range(len(TOKENS[split])) if doc_ids is None else doc_ids
    chunks, lengths, positions = [], [], []
    for d in doc_ids:
        cs = doc_chunks(split, d, setting)
        t = TOKENS[split][d]
        chunks += cs; lengths.append(len(cs))
        positions += [(c, t["n"]) for c in sorted(t["chunks"]) if chosen(c, t["n"], setting)]
    order = sorted(range(len(chunks)), key=lambda i: len(chunks[i]))       # similar lengths together (less padding)
    feats = torch.empty(len(chunks), 2 * HIDDEN)
    for s in range(0, len(order), EVAL_CHUNK_BATCH):
        idx = order[s:s + EVAL_CHUNK_BATCH]
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            feats[idx] = encoder(chunk_batch([chunks[i] for i in idx])).float().cpu()
    return feats, lengths, positions

@torch.no_grad()
def case_vectors(split, setting=EVAL_SETTING, mode="mean"):
    feats, lengths, _ = raw_features(split, setting)
    vecs = model.chunk_vectors(feats.to(DEVICE))
    return pool(vecs, lengths, mode)

def evaluate_vectors(vectors, split, query_ids, depth=100):
    rows = torch.tensor([DOC_INDEX[split][QUERIES[split][q]["key"]] for q in query_ids], device=vectors.device)
    scores = (vectors[rows] @ vectors.T).cpu()
    ranked = rank_queries(scores, split, query_ids, depth)
    return micro_at_k(ranked, gold(split, query_ids)), ranked

def describe(r):
    return f"Micro-F1@7 {r['micro_f1@7']:.4f} (P {r['micro_p@7']:.4f}, R {r['micro_r@7']:.4f}) | Recall@100 {r['recall@100']:.4f}"

def spread(vectors):
    """Mean cosine between random document pairs; close to 1 means every case looks the same (collapse)."""
    i = torch.randint(0, len(vectors), (2000,)); j = torch.randint(0, len(vectors), (2000,))
    return float((vectors[i] * vectors[j]).sum(-1).mean())
print("Encoding helpers ready")

Encoding helpers ready


In [11]:
# Cell 11 — Frozen Phase-1 features (nothing tuned yet) of the CACHE_SETTING chunks of every train / dev / test document, cached.
# Standardisation statistics come from the train chunks; they stay fixed for the rest of the run.
feature_cache = WORK_DIR / "frozen_features.pt"
if not feature_cache.exists():
    previous = sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/frozen_features.pt", recursive=True))
    if previous:
        shutil.copy(previous[0], feature_cache)
if feature_cache.exists():
    FROZEN = torch.load(feature_cache, weights_only=False)
    print("Loaded cached frozen features from", feature_cache)
else:
    start_time = time.time()
    FROZEN = {}
    for split in ("train", "dev", "test"):
        feats, lengths, positions = raw_features(split, CACHE_SETTING)
        FROZEN[split] = {"feats": feats.half(), "lengths": lengths, "positions": positions}
        print(f"  {split}: {len(feats):,} chunks of {len(lengths):,} documents | {(time.time() - start_time) / 60:.1f} min", flush=True)
    torch.save(FROZEN, feature_cache)
feat_mean = FROZEN["train"]["feats"].float().mean(0)
feat_std = FROZEN["train"]["feats"].float().std(0).clamp(min=1e-4)
with torch.no_grad():
    model.feat_mean.copy_(feat_mean); model.feat_std.copy_(feat_std)
print(f"Largest raw feature magnitude {FROZEN['train']['feats'].float().abs().max():.1f} vs mean |x| "
      f"{FROZEN['train']['feats'].float().abs().mean():.2f} → per-dimension standardisation")

def frozen_subset(split, setting):
    """Rows of the cached features that belong to `setting`, and the new chunk count per document."""
    f = FROZEN[split]
    keep = torch.tensor([chosen(c, n, setting) for c, n in f["positions"]])
    lengths, start = [], 0
    for L in f["lengths"]:
        lengths.append(int(keep[start:start + L].sum())); start += L
    return f["feats"][keep].float(), lengths

  train: 40,955 chunks of 4,320 documents | 2.9 min
  dev: 10,200 chunks of 1,023 documents | 3.7 min
  test: 17,111 chunks of 1,727 documents | 5.0 min
Largest raw feature magnitude 8.9 vs mean |x| 0.41 → per-dimension standardisation


In [12]:
# Cell 12 — How Phase-1 representations transfer without any training: frozen standardised features, cosine similarity,
# for each chunk selection and for mean vs max pooling over chunks (dev Micro-F1@7).
def standardised(feats):
    return ((feats - feat_mean) / feat_std).clamp(-FEATURE_CLAMP, FEATURE_CLAMP)

transfer = {}
for setting in SECTION_SETTINGS:
    feats, lengths = frozen_subset("dev", setting)
    x = F.normalize(standardised(feats), dim=-1).to(DEVICE)
    for mode in ("mean", "max"):
        result, _ = evaluate_vectors(pool(x, lengths, mode), "dev", DEV_QUERIES)
        transfer[f"{setting[0]}_{setting[1]}_{mode}"] = result
        print(f"  frozen Phase-1, {setting[0]:9s} {setting[1]:2d} chunks, {mode:4s} pooling: {describe(result)}")
print("TF-IDF reference:", describe(HARD["tfidf_dev"]))

  frozen Phase-1, first      1 chunks, mean pooling: Micro-F1@7 0.0322 (P 0.0351, R 0.0298) | Recall@100 0.2795
  frozen Phase-1, first      1 chunks, max  pooling: Micro-F1@7 0.0322 (P 0.0351, R 0.0298) | Recall@100 0.2795
  frozen Phase-1, first      4 chunks, mean pooling: Micro-F1@7 0.0901 (P 0.0981, R 0.0832) | Recall@100 0.3885
  frozen Phase-1, first      4 chunks, max  pooling: Micro-F1@7 0.0678 (P 0.0738, R 0.0627) | Recall@100 0.3371
  frozen Phase-1, last       4 chunks, mean pooling: Micro-F1@7 0.0689 (P 0.0751, R 0.0637) | Recall@100 0.3546
  frozen Phase-1, last       4 chunks, max  pooling: Micro-F1@7 0.0545 (P 0.0593, R 0.0504) | Recall@100 0.2528
  frozen Phase-1, firstlast  8 chunks, mean pooling: Micro-F1@7 0.1001 (P 0.1090, R 0.0925) | Recall@100 0.4224
  frozen Phase-1, firstlast  8 chunks, max  pooling: Micro-F1@7 0.0723 (P 0.0787, R 0.0668) | Recall@100 0.3289
  frozen Phase-1, firstlast 16 chunks, mean pooling: Micro-F1@7 0.1123 (P 0.1223, R 0.1038) | Recall@100

In [13]:
# Cell 13 — Step 1 (the "LP" of LP-FT): train only the projection on cached frozen features with the contrastive loss
# (InfoNCE: one relevant case + 2 TF-IDF hard negatives per query + in-batch negatives); best on dev = "epoch 0".
TRAIN_PAIRS = [(q, c) for q in TRAIN_QUERIES for c in QUERIES["train"][q]["rel"]]
RELEVANT_KEYS = {q: {DOC_INDEX["train"][c] for c in QUERIES["train"][q]["rel"]} for q in TRAIN_QUERIES}
print(f"{len(TRAIN_PAIRS):,} (query, relevant case) training pairs from {len(TRAIN_QUERIES)} queries")

def contrastive_loss(doc_vecs, batch_docs, pairs):
    """doc_vecs [N, dim] for batch_docs (document indices); pairs = list of (query, relevant case).
    Each query's logits cover every batch document except itself; its other relevant cases are masked."""
    position = {d: i for i, d in enumerate(batch_docs)}
    q_rows = [position[DOC_INDEX["train"][QUERIES["train"][q]["key"]]] for q, _ in pairs]
    targets = torch.tensor([position[DOC_INDEX["train"][c]] for _, c in pairs], device=doc_vecs.device)
    logits = doc_vecs[q_rows] @ doc_vecs.T / TEMPERATURE
    mask = torch.zeros_like(logits, dtype=torch.bool)
    for i, (q, c) in enumerate(pairs):
        mask[i, q_rows[i]] = True
        for d in RELEVANT_KEYS[q]:
            if d in position and position[d] != targets[i].item():
                mask[i, position[d]] = True
    return F.cross_entropy(logits.masked_fill(mask, -1e4), targets)

def pair_batch(pairs, rng):
    """Document indices for a batch: each pair's query, relevant case and HARD_PER_PAIR sampled hard negatives."""
    docs = []
    for q, c in pairs:
        negs = rng.sample(HARD["negatives"][q], min(HARD_PER_PAIR, len(HARD["negatives"][q])))
        for d in [QUERIES["train"][q]["key"], c] + negs:
            i = DOC_INDEX["train"][d]
            if i not in docs:
                docs.append(i)
    return docs

train_feats, train_lengths = frozen_subset("train", EVAL_SETTING)
train_x = standardised(train_feats).to(DEVICE)
offsets = np.concatenate([[0], np.cumsum(train_lengths)])
dev_feats, dev_lengths = frozen_subset("dev", EVAL_SETTING)
dev_x = standardised(dev_feats).to(DEVICE)

def probe_doc_vectors(proj, docs):
    rows = torch.cat([torch.arange(offsets[d], offsets[d + 1]) for d in docs]).to(DEVICE)
    return pool(F.normalize(proj(train_x[rows]), dim=-1), [train_lengths[d] for d in docs])

LATEST, BEST = CKPT_DIR / "latest.pt", CKPT_DIR / "best.pt"
def trainable_state():
    full = model.state_dict()
    return {n: full[n].detach().cpu() for n in SAVED_NAMES}

torch.manual_seed(SEED)
proj = nn.Linear(2 * HIDDEN, PROJ_DIM).to(DEVICE)
nn.init.orthogonal_(proj.weight); nn.init.zeros_(proj.bias)
optimizer = torch.optim.AdamW(proj.parameters(), lr=PROBE_LR, weight_decay=0.01)
rng = random.Random(SEED)
best_probe, probe_history = (-1.0, None, 0), []
start_time = time.time()
for epoch in range(1, PROBE_EPOCHS + 1):
    order = TRAIN_PAIRS[:]; rng.shuffle(order)
    for s in range(0, len(order), PROBE_PAIRS):
        pairs = order[s:s + PROBE_PAIRS]
        docs = pair_batch(pairs, rng)
        loss = contrastive_loss(probe_doc_vectors(proj, docs), docs, pairs)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    if epoch % 3 == 0:
        with torch.no_grad():
            result, _ = evaluate_vectors(pool(F.normalize(proj(dev_x), dim=-1), dev_lengths), "dev", DEV_QUERIES)
        probe_history.append({"epoch": epoch, "loss": round(loss.item(), 4), **result})
        print(f"  probe epoch {epoch:2d}: last loss {loss.item():.3f} | dev {describe(result)}", flush=True)
        if result["micro_f1@7"] > best_probe[0]:
            best_probe = (result["micro_f1@7"], {k: v.detach().clone() for k, v in proj.state_dict().items()}, epoch)
PROBE_F1, probe_state, PROBE_EPOCH = best_probe
with torch.no_grad():                                      # copy the trained projection into the model
    model.proj.weight.copy_(probe_state["weight"]); model.proj.bias.copy_(probe_state["bias"])
probe_dev, _ = evaluate_vectors(case_vectors("dev"), "dev", DEV_QUERIES)
print(f"Projection (probe epoch {PROBE_EPOCH}) copied into the model → epoch 0: {describe(probe_dev)} | "
      f"{(time.time() - start_time) / 60:.1f} min")
assert abs(probe_dev["micro_f1@7"] - PROBE_F1) < 0.02, "The model does not reproduce the probe"
torch.save({"trainable": trainable_state(), "epoch": 0, "dev": probe_dev}, BEST)       # epoch 0 = projection only
del train_x, dev_x, proj
torch.cuda.empty_cache()

5,308 (query, relevant case) training pairs from 817 queries
  probe epoch  3: last loss 3.907 | dev Micro-F1@7 0.1701 (P 0.1852, R 0.1572) | Recall@100 0.5509
  probe epoch  6: last loss 2.696 | dev Micro-F1@7 0.1768 (P 0.1925, R 0.1634) | Recall@100 0.5838
  probe epoch  9: last loss 2.276 | dev Micro-F1@7 0.1857 (P 0.2022, R 0.1716) | Recall@100 0.5663
  probe epoch 12: last loss 1.379 | dev Micro-F1@7 0.1812 (P 0.1973, R 0.1675) | Recall@100 0.5745
  probe epoch 15: last loss 1.513 | dev Micro-F1@7 0.1801 (P 0.1961, R 0.1665) | Recall@100 0.5509
  probe epoch 18: last loss 1.070 | dev Micro-F1@7 0.1812 (P 0.1973, R 0.1675) | Recall@100 0.5694
  probe epoch 21: last loss 1.054 | dev Micro-F1@7 0.1757 (P 0.1913, R 0.1624) | Recall@100 0.5478
  probe epoch 24: last loss 0.902 | dev Micro-F1@7 0.1768 (P 0.1925, R 0.1634) | Recall@100 0.5498
  probe epoch 27: last loss 0.568 | dev Micro-F1@7 0.1745 (P 0.1901, R 0.1614) | Recall@100 0.5252
  probe epoch 30: last loss 0.739 | dev Micro-F1

In [14]:
# Cell 14 — Step 2 (the "FT" of LP-FT): optimiser, learning-rate schedule and checkpoint resume.
adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("encoder.backbone.")]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.backbone.")]   # projection
optimizer = torch.optim.AdamW([{"params": adapter_params, "lr": LR_ADAPTER}, {"params": head_params, "lr": LR_HEAD}],
                              weight_decay=WEIGHT_DECAY)

def epoch_pairs(epoch):
    order = TRAIN_PAIRS[:]
    random.Random(SEED + epoch).shuffle(order)                  # same batches after a resume
    return [order[s:s + PAIRS_PER_STEP] for s in range(0, len(order), PAIRS_PER_STEP)]

steps_per_epoch = len(epoch_pairs(0))
total_updates = EPOCHS * steps_per_epoch
warmup_updates = int(WARMUP_FRACTION * total_updates)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda u: (u + 1) / max(warmup_updates, 1) if u < warmup_updates
                                              else max(0.0, (total_updates - u) / max(total_updates - warmup_updates, 1)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

if not LATEST.exists():
    # Resume only from a run of this notebook with the same saved parameter names.
    for previous in sorted(glob.glob(f"/kaggle/input/**/{RUN_NAME}/checkpoints/latest.pt", recursive=True)):
        if set(torch.load(previous, map_location="cpu", weights_only=False)["trainable"]) == set(SAVED_NAMES):
            shutil.copy(previous, LATEST)
            if Path(previous).with_name("best.pt").exists():
                shutil.copy(Path(previous).with_name("best.pt"), BEST)
            print("Copied checkpoints from", Path(previous).parent)
            break

state = {"epoch": 0, "step": 0, "best_f1": probe_dev["micro_f1@7"], "best_epoch": 0, "history": [], "checks": [],
         "train_seconds": 0.0}
if LATEST.exists():
    ckpt = torch.load(LATEST, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["trainable"], strict=False)
    optimizer.load_state_dict(ckpt["optimizer"]); scheduler.load_state_dict(ckpt["scheduler"]); scaler.load_state_dict(ckpt["scaler"])
    state = ckpt["state"]
    print(f"Resumed at epoch {state['epoch'] + 1}, step {state['step']} (best dev Micro-F1@7 {state['best_f1']:.4f})")
else:
    print("No checkpoint found — fine-tuning starts from the Phase-1 weights + the trained projection.")
print(f"Tuned: {len(adapter_params)} backbone tensors ({sum(p.numel() for p in adapter_params):,} parameters) + projection")
print(f"{steps_per_epoch:,} steps per epoch ({PAIRS_PER_STEP} pairs each), {total_updates:,} optimiser updates in total")

No checkpoint found — fine-tuning starts from the Phase-1 weights + the trained projection.
Tuned: 6 backbone tensors (13,120,000 parameters) + projection
1,327 steps per epoch (4 pairs each), 3,981 optimiser updates in total


In [15]:
# Cell 15 — Fine-tuning the top LSTM layer + projection with the contrastive loss; the LSTM is in train mode (cuDNN needs
# it for the backward pass) with dropout 0. Quick dev checks during the epoch; full dev evaluation after every epoch; best.pt = best of 0…EPOCHS.
def save_latest():
    torch.save({"trainable": trainable_state(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "state": state}, LATEST)

def train_doc_vectors(docs):
    chunks, lengths = [], []
    for d in docs:
        cs = doc_chunks("train", d, TRAIN_SETTING)
        chunks += cs; lengths.append(len(cs))
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        raw = encoder(chunk_batch(chunks))
    return pool(model.chunk_vectors(raw.float()), lengths)

def dev_check():
    vecs = case_vectors("dev")
    result, _ = evaluate_vectors(vecs, "dev", DEV_QUERIES)
    return result, spread(vecs)

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
while state["epoch"] < EPOCHS:
    epoch = state["epoch"]
    batches = epoch_pairs(epoch)
    rng = random.Random(SEED * 1000 + epoch)
    for _ in range(state["step"]):                              # replay the hard-negative draws after a resume
        pair_batch(batches[_], rng)
    model.train()                                               # dropout is 0 everywhere; only tuned layers + projection learn
    start_time, loss_sum, loss_count = time.time(), 0.0, 0
    for step in range(state["step"], len(batches)):
        pairs = batches[step]
        docs = pair_batch(pairs, rng)
        loss = contrastive_loss(train_doc_vectors(docs), docs, pairs)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(adapter_params + head_params, GRAD_CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        loss_sum, loss_count = loss_sum + loss.item(), loss_count + 1
        state["step"] = step + 1
        if state["step"] % SAVE_EVERY_STEPS == 0:
            state["train_seconds"] += time.time() - start_time; start_time = time.time()
            save_latest()
            print(f"  epoch {epoch + 1} step {state['step']}/{len(batches)} | mean loss {loss_sum / loss_count:.4f}", flush=True)
        if state["step"] % CHECK_EVERY_STEPS == 0:
            quick, sp = dev_check()
            state["checks"].append({"epoch": epoch + 1, "step": state["step"], "micro_f1@7": quick["micro_f1@7"], "spread": sp})
            warn = "  WARNING: case vectors nearly identical — collapse" if sp > 0.98 else ""
            print(f"    dev check: {describe(quick)} | mean pair cosine {sp:.3f}{warn}", flush=True)
            model.train(); start_time = time.time()
    state["train_seconds"] += time.time() - start_time
    dev, sp = dev_check()
    state["history"].append({"epoch": epoch + 1, "train_loss": loss_sum / max(loss_count, 1), "dev": dev, "spread": sp})
    mark = ""
    if dev["micro_f1@7"] > state["best_f1"]:
        state.update(best_f1=dev["micro_f1@7"], best_epoch=epoch + 1)
        torch.save({"trainable": trainable_state(), "epoch": epoch + 1, "dev": dev}, BEST)
        mark = "  <- best"
    state["epoch"], state["step"] = epoch + 1, 0
    save_latest()
    print(f"Epoch {epoch + 1} | loss {loss_sum / max(loss_count, 1):.4f} | dev {describe(dev)} | pair cosine {sp:.3f}{mark}", flush=True)
print(f"Best dev Micro-F1@7 {state['best_f1']:.4f} at epoch {state['best_epoch']} (0 = projection only) | "
      f"training time {state['train_seconds'] / 60:.1f} min")

/tmp/ipykernel_23/3399207310.py:38: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


  epoch 1 step 300/1327 | mean loss 1.3501
  epoch 1 step 600/1327 | mean loss 1.5360
    dev check: Micro-F1@7 0.1668 (P 0.1816, R 0.1542) | Recall@100 0.5406 | mean pair cosine 0.666
  epoch 1 step 900/1327 | mean loss 1.5596
  epoch 1 step 1200/1327 | mean loss 1.5610
    dev check: Micro-F1@7 0.1946 (P 0.2119, R 0.1799) | Recall@100 0.5694 | mean pair cosine 0.564
Epoch 1 | loss 1.5476 | dev Micro-F1@7 0.1834 (P 0.1998, R 0.1696) | Recall@100 0.5334 | pair cosine 0.627
  epoch 2 step 300/1327 | mean loss 1.1657
  epoch 2 step 600/1327 | mean loss 1.1388
    dev check: Micro-F1@7 0.1790 (P 0.1949, R 0.1655) | Recall@100 0.5663 | mean pair cosine 0.369
  epoch 2 step 900/1327 | mean loss 1.1408
  epoch 2 step 1200/1327 | mean loss 1.1358
    dev check: Micro-F1@7 0.1857 (P 0.2022, R 0.1716) | Recall@100 0.5714 | mean pair cosine 0.447
Epoch 2 | loss 1.1224 | dev Micro-F1@7 0.1890 (P 0.2058, R 0.1747) | Recall@100 0.5704 | pair cosine 0.390  <- best
  epoch 3 step 300/1327 | mean loss

In [16]:
# Cell 16 — Best model on dev: sections and pooling, score by number of relevant cases, queries without a hit, and how the
# model's hits overlap with TF-IDF's (the reference ranker).
best = torch.load(BEST, map_location="cpu", weights_only=False)
model.load_state_dict(best["trainable"], strict=False)
model.eval()
print(f"Best epoch {best['epoch']} (0 = projection only): {describe(best['dev'])}")

section_results = {}
for setting in SECTION_SETTINGS:
    feats, lengths, _ = raw_features("dev", setting)
    vecs = model.chunk_vectors(feats.to(DEVICE))
    for mode in ("mean", "max"):
        result, _ = evaluate_vectors(pool(vecs, lengths, mode), "dev", DEV_QUERIES)
        section_results[f"{setting[0]}_{setting[1]}_{mode}"] = result
        print(f"  {setting[0]:9s} {setting[1]:2d} chunks, {mode:4s} pooling: {describe(result)}")
BEST_SECTION = max(section_results, key=lambda k: section_results[k]["micro_f1@7"])
kind, k, mode = BEST_SECTION.split("_")
TEST_SETTING, TEST_POOLING = (kind, int(k)), mode
print(f"Best on dev: {BEST_SECTION} → used for the test predictions")

dev_vecs = case_vectors("dev", TEST_SETTING, TEST_POOLING)
dev_result, dev_ranked = evaluate_vectors(dev_vecs, "dev", DEV_QUERIES)
by_bucket = {}
for q, r in zip(DEV_QUERIES, dev_ranked):
    g = set(QUERIES["dev"][q]["rel"]); n = len(g)
    bucket = "1" if n == 1 else "2-3" if n <= 3 else "4-7" if n <= K else "8-15" if n <= 15 else "16+"
    by_bucket.setdefault(bucket, []).append((len(set(r[:K]) & g), n))
dev_by_n = {b: {"queries": len(v), "micro_p@7": round(sum(h for h, _ in v) / (K * len(v)), 4),
                "micro_r@7": round(sum(h for h, _ in v) / sum(n for _, n in v), 4)} for b, v in sorted(by_bucket.items())}
no_hit = [q for q, r in zip(DEV_QUERIES, dev_ranked) if not set(r[:K]) & set(QUERIES["dev"][q]["rel"])]
model_hits = {q: set(r[:K]) & set(QUERIES["dev"][q]["rel"]) for q, r in zip(DEV_QUERIES, dev_ranked)}
tfidf_hits = {q: set(HARD["tfidf_dev_top7"][q]) & set(QUERIES["dev"][q]["rel"]) for q in DEV_QUERIES}
overlap = {"model_hits": sum(len(v) for v in model_hits.values()), "tfidf_hits": sum(len(v) for v in tfidf_hits.values()),
           "both": sum(len(model_hits[q] & tfidf_hits[q]) for q in DEV_QUERIES),
           "union": sum(len(model_hits[q] | tfidf_hits[q]) for q in DEV_QUERIES)}
print("dev by number of relevant cases:", json.dumps(dev_by_n))
print(f"queries without any hit in the top 7: {len(no_hit)} of {len(DEV_QUERIES)}")
print("hits shared with TF-IDF:", overlap)

Best epoch 3 (0 = projection only): Micro-F1@7 0.1979 (P 0.2155, R 0.1829) | Recall@100 0.5899
  first      1 chunks, mean pooling: Micro-F1@7 0.0567 (P 0.0617, R 0.0524) | Recall@100 0.3381
  first      1 chunks, max  pooling: Micro-F1@7 0.0567 (P 0.0617, R 0.0524) | Recall@100 0.3381
  first      4 chunks, mean pooling: Micro-F1@7 0.1623 (P 0.1768, R 0.1501) | Recall@100 0.5262
  first      4 chunks, max  pooling: Micro-F1@7 0.1123 (P 0.1223, R 0.1038) | Recall@100 0.4255
  last       4 chunks, mean pooling: Micro-F1@7 0.1278 (P 0.1392, R 0.1182) | Recall@100 0.4738
  last       4 chunks, max  pooling: Micro-F1@7 0.0800 (P 0.0872, R 0.0740) | Recall@100 0.3854
  firstlast  8 chunks, mean pooling: Micro-F1@7 0.1979 (P 0.2155, R 0.1829) | Recall@100 0.5899
  firstlast  8 chunks, max  pooling: Micro-F1@7 0.1078 (P 0.1174, R 0.0997) | Recall@100 0.4132
  firstlast 16 chunks, mean pooling: Micro-F1@7 0.2168 (P 0.2361, R 0.2004) | Recall@100 0.6156
  firstlast 16 chunks, max  pooling: Micr

In [17]:
# Cell 17 — Predict the test queries (query removed from its own ranking), validate the format, build submission.zip.
# prediction.json = {query id: [7 unique candidate ids from the test pool, most relevant first]} for every test query.
test_vecs = case_vectors("test", TEST_SETTING, TEST_POOLING)
rows = torch.tensor([DOC_INDEX["test"][QUERIES["test"][q]["key"]] for q in TEST_QUERIES], device=test_vecs.device)
test_ranked = rank_queries((test_vecs[rows] @ test_vecs.T).cpu(), "test", TEST_QUERIES, depth=K)
prediction = {q: r[:K] for q, r in zip(TEST_QUERIES, test_ranked)}
pool_set = set(POOL_IDS["test"])
assert set(prediction) == set(QUERIES["test"]), "every test query needs a prediction"
assert all(len(v) == K and len(set(v)) == K and set(v) <= pool_set and q not in v for q, v in prediction.items())
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"))
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
counts = Counter(c for v in prediction.values() for c in v)
print(f"submission.zip: {len(prediction)} test queries × {K} candidates | distinct candidates used {len(counts)} | "
      f"most repeated candidate appears for {counts.most_common(1)[0][1]} queries")

submission.zip: 237 test queries × 7 candidates | distinct candidates used 945 | most repeated candidate appears for 7 queries


In [18]:
# Cell 18 — Final case vectors of every train, dev and test document (TEST_SETTING / TEST_POOLING), with document keys
# (for comparisons and ensembling with the other runs).
start_time = time.time()
teacher = {"setting": list(TEST_SETTING), "pooling": TEST_POOLING, "dim": PROJ_DIM, "dev_result": dev_result}
for split in ("train", "dev", "test"):
    vecs = case_vectors(split, TEST_SETTING, TEST_POOLING) if split != "dev" else dev_vecs
    teacher[split] = {"keys": list(DOC_INDEX[split]), "vectors": vecs.half().cpu()}
torch.save(teacher, OUT_DIR / "teacher_vectors.pt")
check = teacher["dev"]["vectors"].float().to(DEVICE)
again, _ = evaluate_vectors(check, "dev", DEV_QUERIES)
assert abs(again["micro_f1@7"] - dev_result["micro_f1@7"]) < 0.01, "saved teacher vectors do not reproduce the dev score"
print(f"Teacher vectors saved to {OUT_DIR / 'teacher_vectors.pt'} in {time.time() - start_time:.0f} s | dev check {describe(again)}")

Teacher vectors saved to /kaggle/working/P7_S2_C_LSTM_LayerwiseFT/outputs/teacher_vectors.pt in 266 s | dev check Micro-F1@7 0.2168 (P 0.2361, R 0.2004) | Recall@100 0.6156


In [19]:
# Cell 19 — Run summary for the README, the experiment log and results/submissions.csv.
summary = {
    "run": RUN_NAME, "backbone": "Architecture C — stacked LSTM (Phase 1)",
    "technique": "partial / layer-wise FT (top LSTM layer + final LayerNorm) bi-encoder, LP-FT, InfoNCE with TF-IDF hard + in-batch negatives",
    "tuned_lstm_layers": tuned_layers, "trainable_params": n_trainable, "total_params": n_total,
    "train_queries": len(TRAIN_QUERIES), "train_pairs": len(TRAIN_PAIRS), "dev_queries": len(DEV_QUERIES),
    "tfidf_dev_reference": HARD["tfidf_dev"], "frozen_transfer": transfer,
    "probe": {"best_epoch": PROBE_EPOCH, "dev": probe_dev, "history": probe_history},
    "epochs": state["epoch"], "best_epoch": state["best_epoch"], "history": state["history"], "checks": state["checks"],
    "train_minutes": round(state["train_seconds"] / 60, 1),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None,
    "sections": {k: v["micro_f1@7"] for k, v in section_results.items()}, "test_setting": [*TEST_SETTING, TEST_POOLING],
    "dev": dev_result, "dev_by_n_relevant": dev_by_n, "dev_queries_without_hit": len(no_hit), "hits_vs_tfidf": overlap,
}
json.dump(summary, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(json.dumps({k: v for k, v in summary.items() if k not in ("frozen_transfer", "probe", "history", "checks")}, indent=2))

{
  "run": "P7_S2_C_LSTM_LayerwiseFT",
  "backbone": "Architecture C \u2014 stacked LSTM (Phase 1)",
  "technique": "partial / layer-wise FT (top LSTM layer + final LayerNorm) bi-encoder, LP-FT, InfoNCE with TF-IDF hard + in-batch negatives",
  "tuned_lstm_layers": [
    2
  ],
  "trainable_params": 13775616,
  "total_params": 104060496,
  "train_queries": 817,
  "train_pairs": 5308,
  "dev_queries": 118,
  "tfidf_dev_reference": {
    "micro_p@7": 0.4007,
    "micro_r@7": 0.3402,
    "micro_f1@7": 0.368,
    "recall@20": 0.4728,
    "recall@50": 0.5827,
    "recall@100": 0.668
  },
  "epochs": 3,
  "best_epoch": 3,
  "train_minutes": 36.5,
  "peak_gpu_mem_gb": 2.13,
  "sections": {
    "first_1_mean": 0.0567,
    "first_1_max": 0.0567,
    "first_4_mean": 0.1623,
    "first_4_max": 0.1123,
    "last_4_mean": 0.1278,
    "last_4_max": 0.08,
    "firstlast_8_mean": 0.1979,
    "firstlast_8_max": 0.1078,
    "firstlast_16_mean": 0.2168,
    "firstlast_16_max": 0.1034
  },
  "test_setting